<img src="img/ost_logo.png" width="240" align="right"/>
<div style="text-align: left"> <b> Applied Neural Networks: Projekt </b> <br> FS 2026 <br> <a href="mailto:ryan.mikus@ost.ch"> Ryan Mikus </a> und <a href="mailto:ilija.amato@ost.ch"> Ilija Amato </a> </div>

<ul style="list-style:none;padding-left:0;margin:0">
  <li><a href="01_Einleitung.ipynb">1. Einleitung</a></li>
  <li><a href="02_Datenerzeugung.ipynb">2. Datenerzeugung</a></li>
  <li><a href="03_ESN_Baseline_Clean.ipynb">3. ESN-Baseline</a></li>
  <li><a href="04_Rauschen_Robustheit.ipynb">4. Robustheit gegenüber Rauschen</a></li>
  <li><a href="05_Out_of_Distribution.ipynb">5. Out-of-Distribution-Analyse</a></li>
  <li><strong>6. Schlussfolgerung</strong></li>
</ul>

---

# 6. Schlussfolgerung und Ausblick

## 6.1. Gesamtfazit

Dieses Projekt untersuchte Echo State Networks (ESN) für die One-Step-Ahead-Vorhersage gedämpfter harmonischer Schwingungen unter drei Bedingungen: saubere In-Distribution-Daten, additives Rauschen und Out-of-Distribution-Parameter. Die drei Teilfragen lassen sich wie folgt beantworten:

**Teilfrage 1 (Notebook 3): Wie gut funktioniert das ESN auf sauberen Testdaten innerhalb des Trainingsbereichs?**

Sehr gut. Das optimierte Modell (N=400, SR=0.98) erreicht einen mittleren NRMSE von 0.0016, was 0.16 % des Signalbereichs entspricht. Bereits das Baseline-Modell mit Standardhyperparametern (N=300, SR=0.95) liegt bei 0.0024. Beide Modelle lernen die Schwingungsdynamik nahezu fehlerfrei, obwohl nur die lineare Ausgabeschicht $W_{\text{out}}$ trainiert wird.

**Teilfrage 2 (Notebook 4): Wie robust ist das ESN gegenüber Messrauschen?**

Mässig robust. Der NRMSE steigt nahezu linear mit dem Rauschpegel an (Steigung ca. 4.0 NRMSE-Einheiten pro Einheit $\sigma_{\text{rel}}$). Der Leistungsvorteil des optimierten Modells verschwindet bereits bei $\sigma_{\text{rel}} \approx 0.001$, ab diesem Punkt ist die Baseline bei allen getesteten Rauschpegeln besser. Bei $\sigma_{\text{rel}} = 0.1$ erreichen beide Modelle NRMSE-Werte von 0.40 bis 0.44, was auf eine vollständige Sättigung durch das Rauschen hindeutet.

**Teilfrage 3 (Notebook 5): Wie stark bricht die Vorhersagegüte unter OOD-Parametern ein?**

Stark abhängig vom Parameter. Hohe Frequenz ausserhalb des Trainingsbereichs ist der kritischste Faktor (Faktor ~5 gegenüber ID beim optimierten Modell). Hohe Dämpfung ist das einzige Szenario, in dem die Leistung nicht einbricht, weil schnell abklingende Signale konzeptionell einfacher vorherzusagen sind.

**Übergreifendes Ergebnis:** Der Spektralradius ist der zentrale Trade-off-Parameter. Ein hoher Wert ($\rho = 0.98$) ist vorteilhaft für langsam veränderliche, rauscharme Signale im Trainingsbereich. Sobald das Signal schneller wird (hohe Frequenz), schneller abklingt (hohe Dämpfung) oder verrauscht ist, wird das lange Gedächtnis zum Nachteil. Die Baseline mit $\rho = 0.95$ ist in diesen Szenarien konsistent robuster.

| Szenario | Baseline NRMSE | Optimiert NRMSE | Bewertung |
|---|---|---|---|
| Sauber, ID | 0.0024 | 0.0016 | Sehr gut |
| Rauschen $\sigma_{\text{rel}} = 0.01$ | ~0.04 | ~0.05 | Merklicher Einbruch |
| Rauschen $\sigma_{\text{rel}} = 0.1$ | 0.403 | 0.444 | Stark degradiert |
| OOD: Hohe Amplitude | 0.0063 | 0.0047 | Moderat |
| OOD: Hohe Dämpfung | 0.0015 | 0.0017 | Gut (einfacheres Signal) |
| OOD: Hohe Frequenz | 0.0070 | 0.0082 | Kritisch |
| OOD: Kombiniert | 0.0047 | 0.0047 | Moderat |

## 6.2. Kritische Reflexion

Das Projekt liefert klare und konsistente Ergebnisse, hat aber einige methodische Grenzen, die bei der Interpretation berücksichtigt werden sollten.

**Kleiner Trainingsdatensatz.** Mit 25 Trainingssequenzen ist der Datensatz bewusst klein gehalten. Die Ergebnisse sind reproduzierbar (über 10 Seeds validiert), aber eine höhere Sequenzanzahl würde die statistische Aussagekraft erhöhen und klarere Zusammenhänge zwischen Parametern und NRMSE sichtbar machen.

**Ausschliesslich synthetische Daten.** Alle Schwingungen folgen exakt dem Modell $y(t) = A e^{-\alpha t} \sin(\omega t + \phi)$. Reale Messdaten enthalten Modellierungsfehler, Nichtlinearitäten und korreliertes Rauschen. Ob das ESN diese Komplexität ebenfalls gut handhabt, ist nicht untersucht.

**Nur One-Step-Ahead-Vorhersage.** Das Modell gibt immer den nächsten Zeitschritt aus. Autonome Mehrschritt-Prognosen (das Modell speist seine eigene Ausgabe zurück) wurden nicht getestet. Für praktische Anwendungen ist dies oft relevanter.

**Hyperparameter-Optimierung auf zwei Parameter beschränkt.** Die Grid Search variiert nur Reservoir-Grösse $N$ und Spektralradius $\rho$. Weitere Parameter wie Sparsity, Input Scaling und Leaking Rate sind fest. Eine vollständigere Suche würde den Parameterraum besser abdecken.

**Lineare Ausgabeschicht ohne Regularisierung.** $W_{\text{out}}$ wird mit Adam-SGD trainiert. Eine analytische Lösung via Ridge-Regression wäre methodisch direkter und würde L2-Regularisierung explizit einschliessen, was bei kleinen Datensätzen vorteilhaft sein kann.

**Normierung von NRMSE und Rauschen nicht dämpfungsadaptiv.** Sowohl der NRMSE als auch der relative Rauschpegel $\sigma_{\text{rel}}$ werden auf den Gesamtwertebereich $\max(y) - \min(y)$ der Sequenz normiert. Dieser Wert wird durch die Anfangsamplitude bestimmt und bleibt über die gesamte Sequenz konstant. Bei stark gedämpften Schwingungen ist das Signal im späteren Verlauf jedoch nahezu null. Das hat zwei Konsequenzen: Erstens bewertet der NRMSE die triviale Vorhersage "nahezu null" am Sequenzende mit derselben Normierungskonstante wie die schwierigen frühen Zeitschritte, was den Fehler im Mittel unterschätzt. Zweitens entspricht ein fixes $\sigma_{\text{rel}}$ am Sequenzende einem lokal viel höheren Signal-zu-Rausch-Verhältnis als am Anfang, obwohl $\sigma_{\text{rel}}$ konstant erscheint. Eine zeitlokal normierte Metrik, die die Einhüllende $A e^{-\alpha t}$ als Nenner verwendet, würde Sequenzen mit unterschiedlichen Dämpfungsraten vergleichbarer machen. Dieser Effekt erklärt zumindest teilweise, warum das OOD-Szenario "Hohe Dämpfung" als einziges unter dem ID-Referenzwert liegt.

## 6.3. Methodischer Exkurs: Alter vs. neuer Trainingsansatz

Während der Entwicklung dieses Projekts wurde ein grundlegender Methodikfehler entdeckt und korrigiert. Die folgende Abbildung zeigt die Vorhersagen des alten Ansatzes:

<img src="img/alte_ergebnisse.png" width="100%"/>

Das Muster ist eindeutig: Das Modell trifft die **Frequenz** korrekt, aber die **Amplitude** ist systematisch falsch. Statt der abklingenden Einhüllenden $A e^{-\alpha t}$ produziert das ESN eine Schwingung mit näherungsweise konstanter Amplitude.

**Was war der Fehler?**

Im alten Trainingsansatz wurden alle Zeitschritte aller Schwingungen als **unabhängige Einzelsamples** in einen gemeinsamen PyTorch-`Dataset` gepackt und mit `shuffle=True` zufällig in Batches gemischt. Der Reservoir-Zustand $\mathbf{x}$ wurde zu Beginn jedes Batches auf null zurückgesetzt:

```python
# Alter Ansatz (fehlerhaft)
pred, _ = model(u_batch)   # x wird intern auf 0 gesetzt, kein Gedächtnis
```

Das bedeutet: $W_{\text{out}}$ lernte ausschliesslich die **lokale** Abbildung $y(t) \mapsto y(t+1)$, ohne jeglichen Reservoirzustand als Kontext.

**Warum ist die Frequenz trotzdem korrekt?**

Die Frequenz einer gedämpften Schwingung ist im Verhältnis zweier aufeinanderfolgender Werte kodiert:

$$\frac{y(t+1)}{y(t)} \approx e^{-\alpha \Delta t} \cos(\omega \Delta t + \Delta\phi)$$

Diese lokale Beziehung ist unabhängig davon, an welchem Punkt der Zeitreihe man sich befindet. $W_{\text{out}}$ konnte sie allein aus dem aktuellen Input $y(t)$ lernen, ohne Reservoirgedächtnis.

**Warum ist die Amplitude falsch?**

Die Einhüllende $A e^{-\alpha t}$ hängt davon ab, **wie viel Zeit seit dem Beginn der Schwingung vergangen ist**. Dieses globale Wissen ist im Reservoirzustand $\mathbf{x}(t)$ gespeichert, wenn das Reservoir die Sequenz sequenziell verarbeitet. Da $\mathbf{x}$ im alten Ansatz beim Training immer auf null stand, hatte $W_{\text{out}}$ keinen Zugriff auf diese Information. Es lernte stattdessen eine mittlere Amplitude über alle Trainingszeitpunkte, was zu einer nahezu konstanten Hüllkurve führte.

**Der systematische Fehler**

Besonders trügerisch: Der Training-MSE-Loss wirkte akzeptabel (~0.008–0.009), weil der Loss nur lokale Ein-Schritt-Fehler misst. Der RMSE bei sequenzieller Auswertung war hingegen 50- bis 100-mal schlechter als beim korrekten Ansatz. **Die Trainingsmetrik signalisierte Erfolg, obwohl das Modell die Kerneigenschaft des ESN (das Reservoirgedächtnis) gar nicht nutzte.**

**Die Korrektur**

Im neuen Ansatz werden die Reservoir-Zustände **sequenziell gesammelt**: Das Reservoir läuft pro Trainingssequenz von $t=0$ bis $t=T$ durch, der Zustand $\mathbf{x}(t)$ wird bei jedem Schritt weitergegeben. Erst nach diesem Durchlauf werden alle Zustände $(\mathbf{x}(t), y(t+1))$ gesammelt und $W_{\text{out}}$ darauf trainiert.

```python
# Neuer Ansatz (korrekt)
X_train, y_train = eu.collect_states(model, train_dataset, warmup=WARMUP)
eu.train_readout(model, X_train, y_train, ...)
```

Jeder Zustand $\mathbf{x}(t)$ enthält nun die vollständige Geschichte der Schwingung bis zum Zeitpunkt $t$. $W_{\text{out}}$ lernt sowohl die Frequenz als auch die korrekte Amplitude der Einhüllenden zu lesen. Das Ergebnis: NRMSE von ~0.002 statt RMSE von ~0.06–0.25.

## 6.4. Ausblick

Aus den Ergebnissen und der kritischen Reflexion ergeben sich mehrere konkrete Erweiterungsmöglichkeiten.

**Erweiterungen des bestehenden Ansatzes:**

- *Aperiodischer Grenzfall:* Schwingungen mit $\alpha = \omega$ (kritische Dämpfung) liegen genau an der Grenze zwischen Schwingung und exponentiellem Abklingen. Es wäre interessant zu untersuchen, ob das ESN diesen Übergang korrekt erfasst.
- *Ungedämpfte und erzwungene Schwingungen:* Schwingungen mit $\alpha = 0$ oder externem Antriebsterm würden die OOD-Analyse in eine weitere Dimension ausdehnen.
- *Grid Search unter Rauschen und OOD:* Die Hyperparameter wurden ausschliesslich auf sauberen ID-Daten optimiert. Eine separate Grid Search für verrauschte oder OOD-Daten würde zeigen, ob ein robusteres Hyperparameter-Paar existiert, das über alle Szenarien besser abschneidet.
- *Vollständige Grid Search:* Sparsity, Input Scaling und Leaking Rate in die Optimierung einbeziehen.

**Alternative/Vergleichende Modellarchitekturen:**

- *LSTM / GRU:* Gelernte Gating-Mechanismen ermöglichen ein adaptives Gedächtnis, das nicht durch den Spektralradius fest vorgegeben ist. Bei heterogenen Datensätzen (gemischt gedämpft und ungedämpft) wäre dies möglicherweise robuster.
- *Transformer mit Positionskodierung:* Attention-basierte Modelle könnten langfristige Abhängigkeiten in Schwingungssequenzen besser erfassen, benötigen aber deutlich mehr Daten.
- *NARX-Netzwerke (Nonlinear AutoRegressive with eXogenous inputs):* Klassische rückgekoppelte Netze, die explizit vergangene Ausgaben als Eingabe verwenden. Sie sind konzeptionell nah am ESN, aber vollständig trainierbar.
- *Autonome ESN-Prognose:* Statt One-Step-Ahead das ESN im geschlossenen Kreis betreiben, sodass es Schwingungen über mehrere Schritte hinaus autonom weiterführt. Dies würde die Langzeitstabilität des Reservoirs direkt testen.

## 6.5. Literaturverzeichnis

Jaeger, H. (2001). _The "echo state" approach to analysing and training recurrent neural networks_ (GMD Report No. 148). German National Research Center for Information Technology, Bonn.

Jaeger, H., & Haas, H. (2004). Harnessing nonlinearity: Predicting chaotic systems and saving energy in wireless communication. _Science_, _304_(5667), 78–80. https://www.science.org/doi/10.1126/science.1091277

Lukoševičius, M., & Jaeger, H. (2009). Reservoir computing approaches to recurrent neural network training. _Computer Science Review_, _3_(3), 127–149. https://www.sciencedirect.com/science/article/abs/pii/S1574013709000173?via%3Dihub

Lukoševičius, M. (2012). A practical guide to applying echo state networks. In G. Montavon, G. B. Orr, & K.-R. Müller (Eds.), _Neural Networks: Tricks of the Trade_ (2nd ed., pp. 659–686). Springer, Berlin, Heidelberg. https://link.springer.com/chapter/10.1007/978-3-642-35289-8_36

Maass, W., Natschläger, T., & Markram, H. (2002). Real-time computing without stable states: A new framework for neural computation based on perturbations. _Neural Computation_, _14_(11), 2531–2560. https://direct.mit.edu/neco/article-abstract/14/11/2531/6650/Real-Time-Computing-Without-Stable-States-A-New?redirectedFrom=fulltext

Proakis, J. G., & Salehi, M. (2008). _Digital communications_ (5th ed.). McGraw-Hill. Standardreferenz für das AWGN-Rauschmodell in der digitalen Signalverarbeitung.

## 6.6. Verwendete Hilfsmittel

Für die Durchführung dieses Projekts wurden neben den üblichen wissenschaftlichen Quellen folgende KI-gestützte Werkzeuge eingesetzt:

- **Anthropic Claude** ([claude.ai](https://claude.ai)): Methodische Beratung zur Wahl und Konfiguration von Echo State Networks, Unterstützung beim Schreiben und Debuggen des Python/PyTorch-Codes sowie Schreibunterstützung bei der Ausformulierung der Projektdokumentation.
- **OpenAI ChatGPT** ([chatgpt.com](https://chatgpt.com)): Methodische Beratung und Codeunterstützung.
- **GraphvizOnline** ([dreampuf.github.io/GraphvizOnline](https://dreampuf.github.io/GraphvizOnline/)): Browserbasierter Editor zur Erstellung der schematischen Diagramme zur Visualisierung der ESN-Architektur.

Alle KI-generierten Inhalte wurden kritisch geprüft, angepasst und durch eigene Überlegungen ergänzt.

## 6.7. Eigenständigkeitserklärung

«Hiermit bestätigen wir, dass wir die vorliegende Arbeit selbständig verfasst und keine anderen als die angegebenen Hilfsmittel benutzt haben. Die Stellen der Arbeit, die dem Wortlaut oder dem Sinn nach anderen Werken (dazu zählen auch Internetquellen sowie KI-gestützte Werkzeuge) entnommen sind, wurden unter Angabe der Quelle kenntlich gemacht.»

Ryan Mikus &nbsp;&nbsp;&nbsp; Ilija Amato

---

<div style="display: flex; justify-content: flex-start; margin-top: 30px;">
  <a href="05_Out_of_Distribution.ipynb">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Zurück
    </button>
  </a>
</div>